In [1]:
%load_ext autoreload
%autoreload 2

### Match stations, using station name, lat, lon. The name matching in Raw data meta_data form, in the folder and in the database

In [2]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
import sqlalchemy as sa
from ftfy import fix_text

from fern_func import *
# from sql_func import *
from fern_raw_db_dompare import *

In [3]:
# --- Main workflow ---
# --- read data ---
meta_path = '/workspaces/crmprtd/fern/FERNNorth2025_insert/tables/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())


# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/FERNNorth2025_insert/output/'
os.makedirs(output_folder, exist_ok=True)


In [4]:
!pip install ftfy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### Fern raw data condition summary

In [9]:
data_path = '/workspaces/crmprtd/fern/FERNNorth2025_VF_data'
csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
# Sort in natural order
csv_files = natsorted(csv_files)

station_names = [os.path.splitext(f)[0] for f in csv_files]

df_stations = pd.DataFrame(
    {"station_name": station_names}
)

df_stations


summary_fern_raw_records = []

for station_i, csv_file in enumerate(csv_files):
    station_row = df_station.iloc[station_i]
    station_name = station_row[ 'station_name']

    print(station_name)

    ### summarize raw data
    records = summarize_raw_station(csv_file, station_row, data_path)
    summary_fern_raw_records.extend(records)  # add to the master list

raw_summary = pd.DataFrame(summary_fern_raw_records)

raw_summary['unit_origin'] = raw_summary['unit']
raw_summary['unit'] = raw_summary['unit'].astype(str).apply(fix_text)

raw_summary_path = os.path.join(output_folder, "Fern_raw_station_variable_summary.csv")
raw_summary.to_csv(raw_summary_path, index=False)
print(f"✅ Summary saved to {raw_summary_path}")



Atlin School
['Date', 'Day', 'Rain, mm', 'Pressure, mbar', 'Temp, °C', 'RH, %', 'DewPt, °C', 'Wind Speed, m/s', 'Gust Speed, m/s', 'Wind Direction, ø', 'Solar Radiation, W/m²', 'Wetness, %', 'Snow depth, cm', 'Water Content, m³/m³ 15 cm', 'WC_cal, m³/m³', 'Soil Temp, °C', 'Sfc Temp, °C', 'Water Content, m³/m³ 5 cm', 'Water Content, m³/m³ 30 cm', 'Temp 2, °C', 'RH 2, %', 'DewPt 2, °C', 'Temp 30 cm, °C', 'RH 30 cm, %', 'DewPt 30 cm, °C', 'Soil Temp 75 cm, °C', 'Soil Temp 50 cm, °C', 'Flags', 'Comments']
📍 Station: Atlin School (File: Atlin School)
⚠️ Variable 'Pressure' is all NaN, skipping
⚠️ Variable 'Wetness' is all NaN, skipping
⚠️ Variable 'Snow depth' is all NaN, skipping
⚠️ Variable 'Water Content' is all NaN, skipping
⚠️ Variable 'WC_cal' is all NaN, skipping
⚠️ Variable 'Water Content' is all NaN, skipping
⚠️ Variable 'Water Content' is all NaN, skipping
⚠️ Variable 'RH 2' is all NaN, skipping
⚠️ Variable 'DewPt 2' is all NaN, skipping
⚠️ Variable 'Temp 30 cm' is all NaN, skippi

In [7]:
raw_summary

,station_name,native_id,station_file_name,lat,lon,elev,variable,unit,earliest_time,latest_time,unit_origin
0,BulkleyWx,PGTIS1,ChiefLakeWx,53.772183,-122.720729,594,Rain,mm,2019-07-05 14:00:00,2025-09-16 10:00:00,mm
1,BulkleyWx,PGTIS1,ChiefLakeWx,53.772183,-122.720729,594,Pressure,mbar,2019-07-05 13:00:00,2025-09-16 10:00:00,mbar
2,BulkleyWx,PGTIS1,ChiefLakeWx,53.772183,-122.720729,594,Temp,°C,2019-07-05 13:00:00,2025-09-16 10:00:00,°C
3,BulkleyWx,PGTIS1,ChiefLakeWx,53.772183,-122.720729,594,RH,%,2019-07-05 13:00:00,2025-09-16 10:00:00,%
4,BulkleyWx,PGTIS1,ChiefLakeWx,53.772183,-122.720729,594,DewPt,°C,2019-07-05 13:00:00,2025-09-16 10:00:00,°C
...,...,...,...,...,...,...,...,...,...,...,...
76,PinkWx,Pink1,ThompsonWx,57.061630,-122.864910,1756,RH 2,%,2024-09-24 12:00:00,2025-09-22 10:00:00,%
77,PinkWx,Pink1,ThompsonWx,57.061630,-122.864910,1756,DewPt 2,°C,2024-09-24 12:00:00,2025-09-22 10:00:00,°C
78,PinkWx,Pink1,ThompsonWx,57.061630,-122.864910,1756,Temp 30 cm,°C,2007-09-12 18:00:00,2025-09-22 10:00:00,°C
79,PinkWx,Pink1,ThompsonWx,57.061630,-122.864910,1756,RH 30 cm,%,2007-09-12 18:00:00,2025-09-22 10:00:00,%


### Database summary

In [ ]:
import sqlalchemy as sa
from sqlalchemy.orm import Session

engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)


In [ ]:
station_name_id = df_station[['native_id', 'station_name']]
native_ids = station_name_id['native_id'].tolist()
station_names = station_name_id['station_name'].tolist()

query = sa.text("""
    SELECT DISTINCT m.station_name, s.native_id
    FROM meta_history AS m
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11
      AND m.station_name = ANY(:station_names)
""")

with engine.connect() as conn:
    existing_stations = pd.read_sql(query, conn, params={"station_names": station_names})

print("Stations found in DB:")
existing_stations

- Station `Kluskus`  
    - `native_id` in Ted's sheet: `BednestiWx`  
    - `native_id` in Vanessa's sheet: `SBSmc3Wx`

I update the `native_id` in Vanessa's sheet to `BednestiWx`

And we didn't update it according to Vanessa's sheet, so we based on station_name to match stations. All stations can be matched

In [ ]:
existing_native_ids = existing_stations['native_id'].tolist()

query = sa.text("""
    SELECT 
        m.station_name,
        s.native_id,
        m.lat,
        m.lon,
        m.elev,
        v.net_var_name, 
        v.unit,
        MIN(o.obs_time) AS earliest_time,
        MAX(o.obs_time) AS latest_time
    FROM obs_raw AS o
    JOIN meta_history AS m ON o.history_id = m.history_id
    JOIN meta_vars AS v ON o.vars_id = v.vars_id
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11
      AND s.native_id = ANY(:native_ids)
    GROUP BY v.net_var_name, v.unit, m.station_name, m.lat, m.lon, s.native_id, m.elev
    ORDER BY v.net_var_name;
""")

with engine.connect() as conn:
    df_obs = pd.read_sql(query, conn, params={"native_ids": existing_native_ids})

df_obs

In [ ]:
db_summary_path = os.path.join(output_folder, "Fern_db_station_variable_summary.csv")
df_obs.to_csv(db_summary_path, index=False)
print(f"✅ Summary saved to {db_summary_path}")

df_obs